### Data Reading 

In [0]:
df = spark.table("dbacademy.default.big_mart_sales")

df.display()

### Reading CSV File

In [0]:
# dbutils.fs.ls("/Volumes/dbacademy/default/tutorials")

df_json = spark.read.json("/Volumes/dbacademy/default/tutorials/drivers.json")

df_json.display()

# print(df_json.count())



### Schema Definition

In [0]:
df.printSchema()

In [0]:
# To create own schema
my_ddl_schema= '''
Item_Identifier STRING,
Item_Weight STRING,
Item_Fat_Content STRING,
Item_Visibility DOUBLE,
Item_Type STRING,
Item_MRP DOUBLE,
Outlet_Identifier STRING,
Outlet_Establishment_Year LONG,
Outlet_Size STRING,
Outlet_Location_Type STRING,
Outlet_Type STRING,
Item_Outlet_Sales DOUBLE
'''

In [0]:
df = spark.read.format('csv')\
    .schema(my_ddl_schema)\
    .option('header',True)\
    .load('/Volumes/dbacademy/default/tutorials/BigMart Sales.csv')

In [0]:
df.display()

### StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_strct_schema = StructType([ 
                              StructField('Item_Identifier',StringType(),True), 
                              StructField('Item_Weight',StringType(),True), 
                              StructField('Item_Fat_Content',StringType(),True), 
                              StructField('Item_Visibility',StringType(),True), 
                              StructField('Item_Type',StringType(),True), 
                              StructField('Item_MRP',StringType(),True), 
                              StructField('Outlet_Identifier',StringType(),True), 
                              StructField('Outlet_Establishment_Year',StringType(),True), 
                              StructField('Outlet_Size',StringType(),True), 
                              StructField('Outlet_Location_Type',StringType(),True), 
                              StructField('Outlet_Type',StringType(),True), 
                              StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df = spark.read.format('csv')\
    .schema(my_strct_schema)\
    .option('header','true')\
    .load('/Volumes/dbacademy/default/tutorials/BigMart Sales.csv')
df.printSchema()
df.display()
df.count()
df

### SELECT

In [0]:
df.select('Item_Identifier','Item_MRP','Outlet_Size').display()

In [0]:
df.select(col('Item_Identifier'),col('Item_Outlet_Sales'),col('Outlet_Location_Type')).display()

### ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

In [0]:
df.display()

### Filter

#### Scenario 1 - Filter the data with fat content = Regular

In [0]:
df.filter(col("Item_Fat_Content")=="Regular").display()

#### Scenario 2 - Filter Based on Item_Type and Item_Weight

In [0]:
df.filter( (col("Item_Type")=="Soft Drinks") & (col("Item_Weight")<10) ).display()

#### Scenario 3

In [0]:
# df.filter( (col("Outlet_Location_Type")!="Tier 3") & (col("Outlet_Size").isNull()) ).display()
df.filter( (col("Outlet_Location_Type").isin("Tier 1","Tier 2")) & (col("Outlet_Size").isNull()) ).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed("Item_Weight","Item_Wt").display()

### withColumn

#### 1. To create New Column

In [0]:
df = df.withColumn("flag",lit("new")).display()

In [0]:
df.withColumn("mulitply",col("Item_Weight")*col("Item_MRP")).display()

In [0]:
# Replace value "Regular" of table "Item_Fat_Content" with value "Reg"
# df.withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Regular","Reg")).display() 
df.withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Regular","Reg"))\
    .withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Low Fat","Lf")).display()

### Type Casting

In [0]:
# df.withColumn("Item_Fat_Content", col("Item_Fat_Content").cast(StringType()))
df.withColumn("Item_Fat_Content", col("Item_Fat_Content").cast("string"))
df.printSchema()

### Sort

#### Scenario 1

In [0]:
df.sort(col("Item_Weight").desc()).display()

In [0]:
# Sorted table based on item weight and visibility in descending order
df.sort(["Item_Weight","Item_Visibility"],ascending = [0,0]).display()

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending = [0,1]).display()

### Limit

In [0]:
df.limit(4).display()

### Drop

In [0]:
# Drop the specified column
df.drop('Item_Visibility').display()

In [0]:
df.drop("Item_Visibility","Item_Weight").display()

### Drop_Duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
# To remove duplicates from a particular row
df.drop_duplicates(subset=["Outlet_Type"]).display() 

In [0]:
# Show distinct values for all the rows
df.distinct().display()

### UNION and UNION BY NAME

#### Preparing Dataframes

In [0]:
data1 = [('1',"Mansi"),(2,"Nishant")]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1,schema1)

data2 = [('1',"Raj"),('2','Sharma')]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

### UNION

In [0]:
df1.union(df2).display()

In [0]:
data1 = [("Mansi",'1'),("Nishant",'2')]
schema1 = 'name STRING, id STRING'

df1 = spark.createDataFrame(data1,schema1)
df1.display()

### UNION BY NAME

In [0]:
df1.union(df2).display()

# Union the table by checking th name
df1.unionByName(df2).display()

### String Functions

#### Initcap()

In [0]:
# Format the data properly
df.select(initcap("Item_Type")).display()

#### Lower

In [0]:
df.select(lower("Item_Type")).display()

In [0]:
df.select(upper("Item_Type").alias("Upper_Type")).display()

### Date Function

#### Current Date

In [0]:
df = df.withColumn("curr_date",current_date())
df.display()

#### Date_Add()

In [0]:
df = df.withColumn("week_after",date_add("curr_date",7))
df.display()

#### Date_Sub()

In [0]:
df.withColumn("same_date",date_sub("week_after",7)).display()

In [0]:
df = df.withColumn("week_after",date_add("curr_date",-7))

df.display()

#### DateDiff

### DateDiff

In [0]:
df = df.withColumn("date_diff",datediff("week_after","curr_date"))
df.display()


### Date Format

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("week_after",to_date("week_after","dd-MM-yyyy"))
df.display()

In [0]:
df = df.withColumn('week_after',date_format('week_after','dd-MM-yyyy'))

df.display()

### Handling Nulls

#### Dropping Nulls

In [0]:
df.dropna("all").display()

In [0]:
df.dropna("any").display()

In [0]:
df.dropna(subset=["Outlet_Size"]).display()

#### Filling Nulls

In [0]:
df.fillna(0).display()

In [0]:
df.fillna(0,subset=["Item_Weight"]).display()

### SPLIT and Indexing

#### Split

In [0]:
df.withColumn("Outlet_Type",split("Outlet_Type"," ")).display()

#### Indexing

In [0]:
df.withColumn("Outlet_Type",split("Outlet_Type"," ")[1]).display()

### Explode

In [0]:
df_exp = df.withColumn("Outlet_Type",split("Outlet_Type"," "))
df_exp.display()

In [0]:
df_exp.withColumn("Outlet_Type",explode("Outlet_Type")).display()

#### Array_Contains

In [0]:
df_exp = df_exp.withColumn("Type1_flag",array_contains("Outlet_Type","Type1")).display()

#### Group By

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type").agg(avg("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP")).display()

In [0]:
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_MRP"),avg("Item_MRP")).display()

### Collect_List

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy("user").agg(collect_list("book")).display()

#### PIVOT

In [0]:
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

### When-Otherwise

In [0]:
df = df.withColumn("veg-flag",when(col("Item_Type") == "Meat","Non-Veg").otherwise("Veg"))

df.display()

In [0]:
df.withColumn("veg_exp_flag",when((col("veg-flag")=="Veg") & (col("Item_MRP")<100),"Veg_Inexpensive")\
    .when((col("veg-flag")=="Veg") & (col("Item_MRP")>100),"Veg_Expensive")\
    .otherwise("Non-Veg")).display()


## JOINS

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

### INNER JOINS

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'],"inner").display()

### Left Join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],"left").display()

### Right Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],"right").display()

### ANTI-JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],"anti").display()

## Window Functions

### ROW NUMBER()

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn("rowCol",row_number().over(Window.orderBy("Item_Identifier"))).display()

### RANK

In [0]:
df.withColumn('rank',rank().over(Window.orderBy("Item_Identifier"))).display()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col("Item_Identifier").desc()))).display()
# This will first sort the data in descending order and then provide the rank

### Dense_Rank()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()
     

### Cumulative Sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

###  User Defined Functions (UDF)

#### STEP 01 - Define Function

In [0]:
def my_func(x):
    return x*x

#### STEP 02 - Convert Python funcn to pyspark funcn

In [0]:
my_udf = udf(my_func)

#### STEP 03 - Use funcn in pyspark

In [0]:
df.withColumn("mynewcol",my_udf("Item_MRP")).display()

### Data Writing

#### CSV

In [0]:
df.write.format('csv')\
        .save('/Volumes/dbacademy/default/tutorials/big_mart_output')

#### Append

In [0]:
df.write.format('csv')\
    .mode("append")\
    .save('/Volumes/dbacademy/default/tutorials/big_mart_output')

#### Overwrite

In [0]:
df.write.format('csv')\
    .mode("overwrite")\
    .save("/Volumes/dbacademy/default/tutorials/big_mart_output")

#### Error

In [0]:
df.write.format("csv")\
    .mode("error")\
        .save("/Volumes/dbacademy/default/tutorials/big_mart_output")

#### Ignore

In [0]:
df.write.format("csv")\
    .mode("ignore")\
    .save("/Volumes/dbacademy/default/tutorials/big_mart_output")

### Parquet File Format

In [0]:
df.write.format('parquet')\
    .mode("overwrite")\
    .save("/Volumes/dbacademy/default/tutorials/big_mart_output")

#### TABLE

In [0]:
df.write.format('delta')\
    .mode("overwrite")\
    .saveAsTable("my_table")

### SPARK SQL


### createTempView

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view

In [0]:
%sql
select * from my_view where Item_Fat_Content = 'Low Fat'

In [0]:
#### Convert SQL statement output to a Spark DataFrame

df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Low Fat'")

In [0]:
df_sql.display()